In [11]:
import pandas as pd

from utils.utils import get_data
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# X_train, y_train, X_val, y_val, X_test, y_test = get_data(modality='multisource', subset=False, split_strategy='time')

In [ ]:
# mda_vectorizer = TfidfVectorizer(ngram_range=(1,2), max_features=1000, stop_words = 'english', lowercase = True)
# auditor_vectorizer = TfidfVectorizer(ngram_range=(1,2), max_features=1000, stop_words = 'english', lowercase = True)

In [ ]:
# X_mda_train = mda_vectorizer.fit_transform(X_train[1]['item_7'])
# X_auditor_train = auditor_vectorizer.fit_transform(X_train[1]['opinion_text'])

# X_mda_val = mda_vectorizer.transform(X_val[1]['item_7'])
# X_auditor_val = auditor_vectorizer.transform(X_val[1]['opinion_text'])

# X_mda_test = mda_vectorizer.transform(X_test[1]['item_7'])
# X_auditor_test = auditor_vectorizer.transform(X_test[1]['opinion_text'])

In [ ]:
# X_mda_train=X_mda_train.toarray()
# X_mda_val=X_mda_val.toarray()
# X_mda_test=X_mda_test.toarray()

# X_auditor_train=X_auditor_train.toarray()
# X_auditor_val=X_auditor_val.toarray()
# X_auditor_test=X_auditor_test.toarray()

In [ ]:
# from scipy import sparse
# import joblib
# import os

# # Ensure the directory exists
# os.makedirs('data/tfidfs', exist_ok=True)

# # Save matrices
# sparse.save_npz('data/tfidfs/X_mda_train.npz',     X_mda_train)
# sparse.save_npz('data/tfidfs/X_mda_val.npz',       X_mda_val)
# sparse.save_npz('data/tfidfs/X_mda_test.npz',      X_mda_test)

# sparse.save_npz('data/tfidfs/X_auditor_train.npz', X_auditor_train)
# sparse.save_npz('data/tfidfs/X_auditor_val.npz',   X_auditor_val)
# sparse.save_npz('data/tfidfs/X_auditor_test.npz',  X_auditor_test)

# # Save vectorizers
# joblib.dump(mda_vectorizer,     'data/tfidfs/mda_vectorizer.joblib')
# joblib.dump(auditor_vectorizer, 'data/tfidfs/auditor_vectorizer.joblib')


['data/tfidfs/auditor_vectorizer.joblib']

In [1]:
from utils import utils

In [2]:
X_train, y_train, X_val, y_val, X_test, y_test = utils.load_fins_tfidfs()

In [18]:
import numpy as np
import xgboost as xgb
import scipy.sparse
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

class XGBoost():
    def __init__(self, **kwargs):

        self.n_estimators = kwargs['n_estimators']
        self.max_depth = kwargs['max_depth']
        self.learning_rate = kwargs['learning_rate']
        self.device = kwargs.get("device")
 
        self.clf = None  # will be xgb.Booster after training

    @property
    def preprocessor(self) -> BasePreprocessor:
        return self._preprocessor

    def fit(self, X, y):

        y_np = np.array(y)

        dtrain = xgb.DMatrix(X, label=y_np)
        
        params = {
            "max_depth": self.max_depth,
            "learning_rate": self.learning_rate,
            "eval_metric": "logloss",
            "random_state": 42,
            "seed": 42,
            "objective": "binary:logistic",
            "device": self.device,
        }

        self.clf = xgb.train(
            params=params,
            dtrain=dtrain,
            num_boost_round=self.n_estimators,
            verbose_eval=False
        )

    def predict(self, X, threshold =0.5):
        dtest = xgb.DMatrix(X)
        preds = self.clf.predict(dtest)
        return (preds > threshold).astype(int)

    def predict_proba(self, X):
        dtest = xgb.DMatrix(X)
        return self.clf.predict(dtest)

    def evaluate(self, X, y):

        preds = self.predict(X)
        scores = self.predict_proba(X)

        self.auc = roc_auc_score(y_score=scores, y_true=y)
        self.ap = average_precision_score(y_score=scores, y_true=y)

        # recall_at_precision at threshold τ=number of true positives
        num_of_positives = y.sum()
        sorted_indices = np.argsort(scores)[::-1]
        found_labels = y.reset_index(drop=True).loc[sorted_indices]
        
        self.recall_at_precision = (
            found_labels[:num_of_positives].sum() / num_of_positives
        )

        self.recall_at_10 = found_labels[:10].sum() / num_of_positives
        self.recall_at_20 = found_labels[:20].sum() / num_of_positives
        self.recall_at_30 = found_labels[:30].sum() / num_of_positives
        self.recall_at_40 = found_labels[:40].sum() / num_of_positives
        self.recall_at_50 = found_labels[:50].sum() / num_of_positives
        self.recall_at_100 = found_labels[:100].sum() / num_of_positives


        self.auc = round(self.auc, 2)
        self.ap = round(self.ap, 2)

        self.recall_at_precision = round(self.recall_at_precision, 2)
        self.recall_at_10 = round(self.recall_at_10, 2)
        self.recall_at_20 = round(self.recall_at_20, 2)
        self.recall_at_30 = round(self.recall_at_30, 2)
        self.recall_at_40 = round(self.recall_at_40, 2)
        self.recall_at_50 = round(self.recall_at_50, 2)
        self.recall_at_100 = round(self.recall_at_100, 2)

        results = {
            "auc": self.auc,
            "ap": self.ap,
            "recall_at_10" : self.recall_at_10,
            "recall_at_20" : self.recall_at_20,
            "recall_at_30" : self.recall_at_30,
            "recall_at_40" : self.recall_at_40,
            "recall_at_50" : self.recall_at_50,
            "recall_at_100" : self.recall_at_100,
            "recall_at_precision": self.recall_at_precision,
        }

        print(results)

In [24]:
X_tab_train, X_mda_train, X_aud_train = X_train
X_tab_test, X_mda_test, X_aud_test = X_test

In [19]:
model = XGBoost(n_estimators=200, max_depth=8, learning_rate=0.03, device="cuda:0")
model.fit(X_mda_train, y_train)
model.evaluate(X_mda_test, y_test)

{'auc': 0.84, 'ap': 0.19, 'recall_at_10': 0.13, 'recall_at_20': 0.15, 'recall_at_30': 0.17, 'recall_at_40': 0.19, 'recall_at_50': 0.21, 'recall_at_100': 0.28, 'recall_at_precision': 0.21}


In [20]:
model = XGBoost(n_estimators=200, max_depth=8, learning_rate=0.03, device="cuda:0")
model.fit(X_aud_train, y_train)
model.evaluate(X_aud_test, y_test)

{'auc': 0.65, 'ap': 0.03, 'recall_at_10': 0.04, 'recall_at_20': 0.04, 'recall_at_30': 0.04, 'recall_at_40': 0.04, 'recall_at_50': 0.09, 'recall_at_100': 0.11, 'recall_at_precision': 0.09}


In [36]:
model = XGBoost(n_estimators=200, max_depth=8, learning_rate=0.03, device="cuda:0")
X_train = np.concatenate([X_tab_train.values, X_mda_train.toarray(), X_aud_train.toarray()], axis=1)
X_test = np.concatenate([X_tab_test.values, X_mda_test.toarray(), X_aud_test.toarray()], axis=1)
model.fit(X_train, y_train)
model.evaluate(X_test, y_test)

{'auc': 0.91, 'ap': 0.27, 'recall_at_10': 0.13, 'recall_at_20': 0.17, 'recall_at_30': 0.28, 'recall_at_40': 0.32, 'recall_at_50': 0.34, 'recall_at_100': 0.43, 'recall_at_precision': 0.34}
